# Notebook 04 — Analysis and Plots

**Member responsible: C**

Loads all results CSVs from Drive and generates plots for the report.

Plots:
1. Accuracy comparison across all experiments
2. Accuracy vs latency trade-off
3. A2 ablation: accuracy vs LoRA rank
4. A3 ablation: learning curve (accuracy vs training data size)
5. D1 vs D2: effect of filtering on accuracy

In [ ]:
# Step 1a: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 1b: Clone repo (for src imports)
# Paste your GitHub Personal Access Token below (never commit this notebook with the token filled in)
# Generate one at: GitHub > Settings > Developer settings > Personal access tokens > Tokens (classic) > repo scope
import os
token = 'ghp_xxxxxxxxxxxx'  # TODO: paste your token here each session
if not os.path.exists('/content/SC4001-Group133'):
    os.system(f'git clone https://{token}@github.com/TIFFMUN/SC4001-Group133.git /content/SC4001-Group133')
%cd /content/SC4001-Group133
!pip install -r requirements.txt -q

In [ ]:
# Step 1c: Redirect cache so results/ points to Drive CSVs
import sys
sys.path.insert(0, '/content/SC4001-Group133')
from src.setup_cache import setup
setup()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.evaluation.metrics import compute_summary_stats

sns.set_theme(style='whitegrid', palette='muted')
RESULTS_DIR = Path('results/')
PLOT_DIR = Path('results/plots/')
PLOT_DIR.mkdir(exist_ok=True)

def load_and_summarise(csv_path, exp_name):
    df = pd.read_csv(csv_path)
    stats = compute_summary_stats(df.to_dict(orient='records'))
    stats['experiment'] = exp_name
    return stats

# TODO: add/remove rows as experiments complete
experiment_files = [
    ('exp0_zero_shot_gsm8k.csv',      'Exp 0: Zero-shot'),
    ('exp1_few_shot_gsm8k.csv',       'Exp 1: Few-shot CoT'),
    ('exp2_adaptation_r16_gsm8k.csv', 'Exp 2: Adaptation (r=16)'),
    ('exp3_distillation_gsm8k.csv',   'Exp 3: Distillation D1'),
    ('exp4_filtered_gsm8k.csv',       'Exp 4: Filtered D2'),
    ('exp5_d3_cross_task_arc.csv',    'Exp 5: Cross-task ARC'),
]

summaries = []
for fname, exp_name in experiment_files:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        summaries.append(load_and_summarise(str(fpath), exp_name))
    else:
        print(f'[SKIP] {fname} not found yet')

summary_df = pd.DataFrame(summaries)
summary_df

In [ ]:
# Plot 1: Accuracy comparison across all experiments
fig, ax = plt.subplots(figsize=(10, 5))
bars = sns.barplot(data=summary_df, x='experiment', y='accuracy', ax=ax)
ax.set_title('Accuracy by Experiment', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
for bar, row in zip(bars.patches, summary_df.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{row.accuracy:.2%}', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot1_accuracy_comparison.png', dpi=150)
plt.show()

In [ ]:
# Plot 2: Accuracy vs latency trade-off (key figure for the report)
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=summary_df, x='mean_latency_ms', y='accuracy', s=120, ax=ax)
for _, row in summary_df.iterrows():
    ax.annotate(row['experiment'], (row['mean_latency_ms'], row['accuracy']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)
ax.set_title('Accuracy vs Latency Trade-off', fontsize=13)
ax.set_xlabel('Mean Latency (ms/sample)')
ax.set_ylabel('Accuracy')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot2_accuracy_vs_latency.png', dpi=150)
plt.show()

In [ ]:
# Plot 3: A2 ablation — accuracy vs LoRA rank
# TODO: fill in actual values after running A2 ablation (rank sweep in notebook 02)
a2_data = pd.DataFrame({
    'rank': [4, 8, 16, 32],
    'accuracy': [None, None, None, None],        # TODO: fill after runs
    'mean_latency_ms': [None, None, None, None], # TODO: fill after runs
})

fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()
ax1.plot(a2_data['rank'], a2_data['accuracy'], marker='o', color='steelblue', label='Accuracy')
ax2.plot(a2_data['rank'], a2_data['mean_latency_ms'], marker='s', color='coral',
         linestyle='--', label='Latency')
ax1.set_xlabel('LoRA Rank (r)')
ax1.set_ylabel('Accuracy', color='steelblue')
ax2.set_ylabel('Mean Latency (ms)', color='coral')
ax1.set_title('A2 Ablation: Accuracy & Latency vs LoRA Rank')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot3_a2_rank_ablation.png', dpi=150)
plt.show()

In [ ]:
# Plot 4: A3 ablation — learning curve
# TODO: fill in actual values after running A3 ablation (data size sweep in notebook 02)
a3_data = pd.DataFrame({
    'fraction': [0.1, 0.25, 0.5, 1.0],
    'n_samples': [747, 1868, 3736, 7473],
    'accuracy': [None, None, None, None],  # TODO: fill after runs
})

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(a3_data['n_samples'], a3_data['accuracy'], marker='o', color='mediumseagreen')
ax.set_xlabel('Training Samples')
ax.set_ylabel('Accuracy')
ax.set_title('A3 Ablation: Learning Curve (Accuracy vs Training Size)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot4_a3_data_size_ablation.png', dpi=150)
plt.show()

In [ ]:
# Save full summary table as CSV for the report
summary_df.to_csv(RESULTS_DIR / 'summary_all_experiments.csv', index=False)
print('Summary saved to results/summary_all_experiments.csv')
summary_df[['experiment', 'accuracy', 'mean_latency_ms', 'mean_token_count', 'n_samples']]